<a href="https://colab.research.google.com/github/FyyyyL/python-notes/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================
# EDA SECTION: NEWS2 trajectories vs Sepsis
# (Result-oriented EDA for thesis figures)
# ======================================================

# ===============================
# STEP 0: Mount Google Drive
# ===============================
from google.colab import drive
drive.mount('/content/drive')

# ===============================
# STEP 1: Import libraries
# ===============================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# ===============================
# STEP 2: File paths
# ===============================
INPUT_PATH = "/content/drive/MyDrive/FinalYearProject/Outputs/Sepsis_with_NEWS2.csv"
OUTPUT_DIR = "/content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===============================
# STEP 3: Load and sort data
# ===============================
df = pd.read_csv(INPUT_PATH)
df = df.sort_values(["Patient_ID", "Hour"]).reset_index(drop=True)

print("Loaded data shape:", df.shape)

# ======================================================
# FIGURE 1: Representative NEWS2 trajectories
# ======================================================
np.random.seed(2025)

sepsis_ids = df.loc[df["SepsisLabel"] == 1, "Patient_ID"].unique()
nonsepsis_ids = df.loc[df["SepsisLabel"] == 0, "Patient_ID"].unique()

rep_sepsis = np.random.choice(sepsis_ids, size=min(6, len(sepsis_ids)), replace=False)
rep_nonsepsis = np.random.choice(nonsepsis_ids, size=min(6, len(nonsepsis_ids)), replace=False)

plt.figure(figsize=(12, 6))

for pid in rep_sepsis:
    sub = df[df["Patient_ID"] == pid]
    plt.plot(sub["Hour"], sub["NEWS2_score"], alpha=0.7)

for pid in rep_nonsepsis:
    sub = df[df["Patient_ID"] == pid]
    plt.plot(sub["Hour"], sub["NEWS2_score"], alpha=0.7, linestyle="--")

plt.xlabel("ICU Hour")
plt.ylabel("NEWS2 Score")
plt.title("Representative NEWS2 Trajectories\nSolid = Sepsis, Dashed = Non-sepsis")
plt.tight_layout()

fig1_path = os.path.join(OUTPUT_DIR, "fig1_news2_representative_trajectories.png")
plt.savefig(fig1_path, dpi=300)
plt.close()

print("Saved:", fig1_path)

# ======================================================
# FIGURE 2: Mean NEWS2 trajectory (first 72 hours)
# ======================================================
df_72 = df[df["Hour"] <= 72]

mean_traj = (
    df_72
    .groupby(["SepsisLabel", "Hour"])["NEWS2_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))

for label, style in [(0, "--"), (1, "-")]:
    sub = mean_traj[mean_traj["SepsisLabel"] == label]
    plt.plot(sub["Hour"], sub["NEWS2_score"], linestyle=style)

plt.xlabel("ICU Hour")
plt.ylabel("Mean NEWS2 Score")
plt.title("Mean NEWS2 Trajectories (First 72 ICU Hours)")
plt.legend(["Non-sepsis", "Sepsis"])
plt.tight_layout()

fig2_path = os.path.join(OUTPUT_DIR, "fig2_mean_news2_trajectory_72h.png")
plt.savefig(fig2_path, dpi=300)
plt.close()

print("Saved:", fig2_path)

# ======================================================
# FIGURE 3: Distribution of NEWS2 scores by sepsis status
# ======================================================
plt.figure(figsize=(8, 6))

plt.boxplot(
    [
        df[df["SepsisLabel"] == 0]["NEWS2_score"],
        df[df["SepsisLabel"] == 1]["NEWS2_score"]
    ],
    labels=["Non-sepsis", "Sepsis"],
    showfliers=False
)

plt.ylabel("NEWS2 Score")
plt.title("Distribution of NEWS2 Scores by Sepsis Status")
plt.tight_layout()

fig3_path = os.path.join(OUTPUT_DIR, "fig3_news2_distribution_by_sepsis.png")
plt.savefig(fig3_path, dpi=300)
plt.close()

print("Saved:", fig3_path)

# ======================================================
# FIGURE 4: Patient-level NEWS2 summaries
# ======================================================
patient_summary = (
    df
    .groupby(["Patient_ID", "SepsisLabel"])
    .agg(
        max_news2=("NEWS2_score", "max"),
        mean_news2=("NEWS2_score", "mean"),
        p95_news2=("NEWS2_score", lambda x: np.percentile(x, 95))
    )
    .reset_index()
)

plt.figure(figsize=(12, 4))

for i, col in enumerate(["max_news2", "mean_news2", "p95_news2"], start=1):
    plt.subplot(1, 3, i)
    plt.boxplot(
        [
            patient_summary[patient_summary["SepsisLabel"] == 0][col],
            patient_summary[patient_summary["SepsisLabel"] == 1][col]
        ],
        labels=["Non-sepsis", "Sepsis"],
        showfliers=False
    )
    plt.title(col.replace("_", " ").upper())

plt.suptitle("Patient-level NEWS2 Summaries by Sepsis Status")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

fig4_path = os.path.join(OUTPUT_DIR, "fig4_patient_level_news2_summaries.png")
plt.savefig(fig4_path, dpi=300)
plt.close()

print("Saved:", fig4_path)

# ===============================
# Save patient-level summary table
# ===============================
summary_csv_path = os.path.join(OUTPUT_DIR, "patient_level_news2_summary.csv")
patient_summary.to_csv(summary_csv_path, index=False)

print("Saved:", summary_csv_path)

print("\nEDA (NEWS2 trajectories vs sepsis) completed successfully.")

Mounted at /content/drive
Loaded data shape: (1552210, 53)
Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/fig1_news2_representative_trajectories.png
Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/fig2_mean_news2_trajectory_72h.png


/tmp/ipython-input-115512003.py:103: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(


Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/fig3_news2_distribution_by_sepsis.png


/tmp/ipython-input-115512003.py:140: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(
/tmp/ipython-input-115512003.py:140: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(
/tmp/ipython-input-115512003.py:140: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(


Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/fig4_patient_level_news2_summaries.png
Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/patient_level_news2_summary.csv

EDA (NEWS2 trajectories vs sepsis) completed successfully.


In [ ]:
# ======================================================
# EDA RESULT FIGURES: Background trajectories + bold mean
# First 72 hours, split by sepsis status
# Outputs saved to Google Drive
# ======================================================

# -------------------------------
# STEP 0: Mount Google Drive (Colab only)
# -------------------------------
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# STEP 1: Imports
# -------------------------------
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# -------------------------------
# STEP 2: Paths
# -------------------------------
INPUT_PATH = "/content/drive/MyDrive/FinalYearProject/Outputs/Sepsis_with_NEWS2.csv"
OUTPUT_DIR = "/content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------
# STEP 3: Load + sort + restrict to first 72h
# -------------------------------
df = pd.read_csv(INPUT_PATH)
df = df.sort_values(["Patient_ID", "Hour"]).reset_index(drop=True)

H_MAX = 72
df = df[df["Hour"] <= H_MAX].copy()

# Basic column check
required = ["Patient_ID", "Hour", "SepsisLabel", "NEWS2_score"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -------------------------------
# STEP 4: Helper function
# -------------------------------
def plot_background_plus_mean(
    df_group,
    title,
    outpath,
    max_patients_bg=400,
    seed=2025
):
    """
    Background: many patient trajectories (very transparent)
    Foreground: mean NEWS2 trajectory (bold)
    """
    # Cap number of background trajectories (keeps plot readable + file size reasonable)
    pids = df_group["Patient_ID"].unique()
    rng = np.random.default_rng(seed)
    if len(pids) > max_patients_bg:
        pids_bg = rng.choice(pids, size=max_patients_bg, replace=False)
    else:
        pids_bg = pids

    plt.figure(figsize=(12, 6))

    # Background individual trajectories
    for pid in pids_bg:
        sub = df_group[df_group["Patient_ID"] == pid]
        plt.plot(sub["Hour"], sub["NEWS2_score"], alpha=0.05, linewidth=0.8)

    # Mean trajectory (bold)
    mean_traj = df_group.groupby("Hour")["NEWS2_score"].mean().reset_index()
    plt.plot(mean_traj["Hour"], mean_traj["NEWS2_score"], linewidth=3.0)

    plt.xlabel("ICU Hour (0–72)")
    plt.ylabel("NEWS2 Score")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

# -------------------------------
# STEP 5: Split groups and plot
# -------------------------------
df_sepsis = df[df["SepsisLabel"] == 1].copy()
df_non = df[df["SepsisLabel"] == 0].copy()

out_sepsis = os.path.join(OUTPUT_DIR, "figA_background_plus_mean_sepsis_72h.png")
out_non = os.path.join(OUTPUT_DIR, "figB_background_plus_mean_nonsepsis_72h.png")

plot_background_plus_mean(
    df_sepsis,
    "NEWS2 Trajectories (Sepsis): Background Patients + Mean Trend (First 72 ICU Hours)",
    out_sepsis,
    max_patients_bg=400
)

plot_background_plus_mean(
    df_non,
    "NEWS2 Trajectories (Non-sepsis): Background Patients + Mean Trend (First 72 ICU Hours)",
    out_non,
    max_patients_bg=400
)

print("Saved:", out_sepsis)
print("Saved:", out_non)
print("EDA figures done.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/figA_background_plus_mean_sepsis_72h.png
Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/figB_background_plus_mean_nonsepsis_72h.png
EDA figures done.


In [ ]:
# ======================================================
# EDA RESULT FIGURE
# Background trajectories + bold mean
# ONE figure, two panels: Sepsis vs Non-sepsis
# First 72 ICU hours
# ======================================================

# -------------------------------
# STEP 0: Mount Google Drive (Colab only)
# -------------------------------
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# STEP 1: Imports
# -------------------------------
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# -------------------------------
# STEP 2: Paths
# -------------------------------
INPUT_PATH = "/content/drive/MyDrive/FinalYearProject/Outputs/Sepsis_with_NEWS2.csv"
OUTPUT_DIR = "/content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------
# STEP 3: Load + sort + restrict to first 72h
# -------------------------------
df = pd.read_csv(INPUT_PATH)
df = df.sort_values(["Patient_ID", "Hour"]).reset_index(drop=True)

H_MAX = 72
df = df[df["Hour"] <= H_MAX].copy()

# Safety check
required = ["Patient_ID", "Hour", "SepsisLabel", "NEWS2_score"]
for c in required:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# -------------------------------
# STEP 4: Helper function (plot on given axis)
# -------------------------------
def plot_background_plus_mean(
    ax,
    df_group,
    title,
    max_patients_bg=400,
    seed=2025
):
    """
    Background: many patient trajectories (very transparent)
    Foreground: mean NEWS2 trajectory (bold)
    """
    pids = df_group["Patient_ID"].unique()
    rng = np.random.default_rng(seed)

    if len(pids) > max_patients_bg:
        pids_bg = rng.choice(pids, size=max_patients_bg, replace=False)
    else:
        pids_bg = pids

    # Background individual trajectories
    for pid in pids_bg:
        sub = df_group[df_group["Patient_ID"] == pid]
        ax.plot(sub["Hour"], sub["NEWS2_score"],
                alpha=0.05, linewidth=0.8)

    # Mean trajectory
    mean_traj = (
        df_group
        .groupby("Hour")["NEWS2_score"]
        .mean()
        .reset_index()
    )
    ax.plot(mean_traj["Hour"], mean_traj["NEWS2_score"],
            linewidth=3.0)

    ax.set_title(title)
    ax.set_xlabel("ICU Hour (0–72)")
    ax.set_ylabel("NEWS2 Score")

# -------------------------------
# STEP 5: Split groups
# -------------------------------
df_sepsis = df[df["SepsisLabel"] == 1].copy()
df_non = df[df["SepsisLabel"] == 0].copy()

# -------------------------------
# STEP 6: Create combined figure
# -------------------------------
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6),
    sharey=True
)

plot_background_plus_mean(
    axes[0],
    df_sepsis,
    title="Sepsis"
)

plot_background_plus_mean(
    axes[1],
    df_non,
    title="Non-sepsis"
)

fig.suptitle(
    "NEWS2 Trajectories: Background Patients and Mean Trend\n(First 72 ICU Hours)",
    fontsize=14
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# -------------------------------
# STEP 7: Save figure
# -------------------------------
out_path = os.path.join(
    OUTPUT_DIR,
    "fig_background_plus_mean_sepsis_vs_nonsepsis_72h.png"
)

plt.savefig(out_path, dpi=300)
plt.close()

print("Saved:", out_path)
print("Combined EDA figure completed.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/MyDrive/FinalYearProject/Outputs/EDA_NEWS2/fig_background_plus_mean_sepsis_vs_nonsepsis_72h.png
Combined EDA figure completed.
